In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import joblib
from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, precision_recall_fscore_support, confusion_matrix, average_precision_score
import json

paths = load_paths()
logger = setup_logger(level="INFO")

# Load Data and Predictions
logger.info("Loading data and predictions...")

# UPDATED: Load predictions from the Balanced Bagging output
bagging_dir = paths.artifacts_dir / "balanced_bagging"
preds_path = bagging_dir / "predictions.csv" # Changed from test_predictions.csv

if not preds_path.exists():
    raise FileNotFoundError(f"Predictions not found at {preds_path}. Run Notebook 10 first.")

df_preds = pd.read_csv(preds_path)

if "prob" in df_preds.columns:
    df_preds = df_preds.rename(columns={"prob": "p_calib"})
elif "p_calib" not in df_preds.columns:
    raise ValueError("Missing calibrated probability column: expected 'prob' or 'p_calib'")

if "prob_raw" not in df_preds.columns:
    logger.warning("Raw probability column 'prob_raw' not found. Raw-vs-calibrated comparison will be limited.")

# Load Feature Pipeline to get feature names
feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline = FeaturePipeline.load(feature_art)
all_pipeline_cols = pipeline.model_feature_names()

# Filter feature columns to match what the model likely used (numeric only, no metadata)
# This replicates get_feature_cols from train_balanced_bagging_ensemble.py
exclude_cols = {"label", "capture_id", "dataset", "split", "flow_id", "timestamp", "src_ip", "dst_ip", "src_port", "dst_port", "protocol", "sample_weight"}
feature_cols = [c for c in all_pipeline_cols if c not in exclude_cols]

# Load the raw features to analyze correlations
# We need to load the features to get the feature values for the test set rows
logger.info("Loading feature data...")

def load_features_safe(dataset_name):
    p = paths.data_processed / dataset_name / "features.parquet"
    if not p.exists():
        # Try fallback
        p = paths.data_processed / dataset_name / "features_trainable.parquet"

    if p.exists():
        df = pd.read_parquet(p)
        df["dataset"] = dataset_name
        return df
    else:
        logger.warning(f"Feature file for {dataset_name} not found. Skipping feature analysis for this dataset.")
        return None

vnat_feats = load_features_safe("vnat")
iscx_feats = load_features_safe("iscx")

if vnat_feats is not None and iscx_feats is not None:
    df_features = pd.concat([vnat_feats, iscx_feats], ignore_index=True)
    has_features = True
elif vnat_feats is not None:
    df_features = vnat_feats
    has_features = True
elif iscx_feats is not None:
    df_features = iscx_feats
    has_features = True
else:
    df_features = pd.DataFrame()
    has_features = False
    logger.warning("No feature files found. Skipping feature-level analysis.")

print(f"Predictions Shape: {df_preds.shape}")

# --- RAW vs CALIBRATED COMPARISON ---
from sklearn.metrics import brier_score_loss, log_loss

if "prob_raw" in df_preds.columns:
    print("\n--- RAW vs CALIBRATED PERFORMANCE (TEST) ---")
    test_df_cmp = df_preds[df_preds["split"] == "test"].copy()

    y_cmp = test_df_cmp["label"].astype(int)

    auc_raw = roc_auc_score(y_cmp, test_df_cmp["prob_raw"])
    auc_cal = roc_auc_score(y_cmp, test_df_cmp["p_calib"])

    ap_raw = average_precision_score(y_cmp, test_df_cmp["prob_raw"])
    ap_cal = average_precision_score(y_cmp, test_df_cmp["p_calib"])

    brier_raw = brier_score_loss(y_cmp, test_df_cmp["prob_raw"])
    brier_cal = brier_score_loss(y_cmp, test_df_cmp["p_calib"])

    ll_raw = log_loss(y_cmp, np.clip(test_df_cmp["prob_raw"], 1e-6, 1 - 1e-6))
    ll_cal = log_loss(y_cmp, np.clip(test_df_cmp["p_calib"], 1e-6, 1 - 1e-6))

    print(f"ROC AUC   raw={auc_raw:.4f}  calib={auc_cal:.4f}")
    print(f"PR AUC    raw={ap_raw:.4f}  calib={ap_cal:.4f}")
    print(f"Brier     raw={brier_raw:.4f}  calib={brier_cal:.4f}")
    print(f"LogLoss   raw={ll_raw:.4f}  calib={ll_cal:.4f}")


2026-03-27 13:05:05 | INFO | ai-vpn-firewall | Loading data and predictions...
2026-03-27 13:05:06 | INFO | ai-vpn-firewall | Loading feature data...
Predictions Shape: (69558, 12)

--- RAW vs CALIBRATED PERFORMANCE (TEST) ---
ROC AUC   raw=0.9645  calib=0.9625
PR AUC    raw=0.8645  calib=0.8358
Brier     raw=0.0528  calib=0.0535
LogLoss   raw=0.1652  calib=0.1736


In [2]:
# 1. Feature Importance Analysis (using one of the Bagged XGBoost models)
logger.info("Analyzing Feature Importance...")
# Load the first XGBoost bag model
model_path = bagging_dir / "model_xgb_bag0.pkl"

if model_path.exists():
    model = joblib.load(model_path)

    # XGBClassifier exposes feature_importances_
    # We need to map them to feature names
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_

        feature_cols_plot = feature_cols.copy()

        if len(importances) != len(feature_cols_plot):
            print(f"DEBUG: Mismatch detected. Model has {len(importances)}, Pipeline has {len(feature_cols_plot)}.")

            if len(importances) == len(feature_cols_plot) + 1 and "q_min_packets_ok" not in feature_cols_plot:
                print("DEBUG: Attempting to patch with 'q_min_packets_ok'...")
                feature_cols_plot.append("q_min_packets_ok")

        if len(importances) == len(feature_cols_plot):
            imp_df = pd.DataFrame({
                "Feature": feature_cols_plot,
                "Gain": importances
            }).sort_values("Gain", ascending=False)
        else:
            print(f"Still mismatched after patch. Model has {len(importances)}, feature list has {len(feature_cols_plot)}.")
            imp_df = None

        if imp_df is not None:
            plt.figure(figsize=(10, 8))
            sns.barplot(x="Gain", y="Feature", data=imp_df.head(20))
            plt.title("Top 20 Features by Importance (XGBoost Bag 0)")
            plt.tight_layout()
            plt.show()

            print("Top 5 Features:")
            print(imp_df.head(5))

            # Check for "Dominator" features
            top_gain = imp_df.iloc[0]["Gain"]
            total_gain = imp_df["Gain"].sum()
            print(f"\nTop feature '{imp_df.iloc[0]['Feature']}' accounts for {top_gain/total_gain:.2%} of total importance.")
            if top_gain/total_gain > 0.5:
                print("WARNING: Single feature dominance detected! Potential leakage.")
    else:
        print("Model does not have feature_importances_ attribute.")
else:
    print(f"Model file not found: {model_path}")


2026-03-27 13:05:06 | INFO | ai-vpn-firewall | Analyzing Feature Importance...
DEBUG: Mismatch detected. Model has 9, Pipeline has 27.
Still mismatched after patch. Model has 9, feature list has 27.


In [3]:
# 2. Cross-Dataset Performance (VNAT vs ISCX)
logger.info("Analyzing Cross-Dataset Performance on TEST set...")
# Filter for TEST set
test_df = df_preds[df_preds["split"] == "test"].copy()

# Load thresholds from metrics.json
metrics_path = bagging_dir / "metrics.json"
if metrics_path.exists():
    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    # Helper to get threshold
    def get_thr(metrics_dict, fpr):
        key = f"fpr_{fpr}"
        if key in metrics_dict: return metrics_dict[key]["threshold"]
        return 0.99

    val_metrics = metrics.get("val", {})
    T_BLOCK = get_thr(val_metrics, 0.001)
    T_MONITOR = get_thr(val_metrics, 0.01)

    print(f"Global Thresholds: BLOCK={T_BLOCK:.4f}, MONITOR={T_MONITOR:.4f}")

    for ds in ["vnat", "iscx"]:
        subset = test_df[test_df["dataset"] == ds]
        if len(subset) == 0: continue

        auc_val = roc_auc_score(subset["label"], subset["p_calib"])
        print(f"\nDataset: {ds.upper()} (N={len(subset)})")
        print(f"  ROC AUC: {auc_val:.4f}")
        print(f"  Mean Prob (VPN): {subset[subset['label']==1]['p_calib'].mean():.4f}")
        print(f"  Mean Prob (Non-VPN): {subset[subset['label']==0]['p_calib'].mean():.4f}")

        for zone, t in [("BLOCK", T_BLOCK), ("MONITOR", T_MONITOR)]:
            y_hat = (subset["p_calib"] >= t).astype(int)
            prec, rec, _, _ = precision_recall_fscore_support(subset["label"], y_hat, average="binary", zero_division=0)
            _, fp, _, _ = confusion_matrix(subset["label"], y_hat).ravel()
            print(f"  {zone} (p >= {t:.4f}): Recall={rec:.4f}, Precision={prec:.4f}, FP={fp}")
else:
    print("Metrics file not found. Skipping threshold analysis.")


2026-03-27 13:05:06 | INFO | ai-vpn-firewall | Analyzing Cross-Dataset Performance on TEST set...
Global Thresholds: BLOCK=0.9900, MONITOR=0.9900

Dataset: VNAT (N=210)
  ROC AUC: 1.0000
  Mean Prob (VPN): 0.8684
  Mean Prob (Non-VPN): 0.3171
  BLOCK (p >= 0.9900): Recall=0.0000, Precision=0.0000, FP=0
  MONITOR (p >= 0.9900): Recall=0.0000, Precision=0.0000, FP=0

Dataset: ISCX (N=1634)
  ROC AUC: 0.7119
  Mean Prob (VPN): 0.5151
  Mean Prob (Non-VPN): 0.3817
  BLOCK (p >= 0.9900): Recall=0.0000, Precision=0.0000, FP=0
  MONITOR (p >= 0.9900): Recall=0.0000, Precision=0.0000, FP=0


In [4]:
# 3. "Easy" vs "Hard" Traffic Analysis
logger.info("Analyzing Hard Cases...")
test_df["error"] = np.abs(test_df["label"] - test_df["p_calib"])
hard_cases = test_df[test_df["error"] > 0.5]

print(f"\nNumber of Misclassified Test Samples: {len(hard_cases)} out of {len(test_df)}")

if len(hard_cases) > 0:
    print("\nTop Misclassified Samples (by error):")
    print(hard_cases.sort_values("error", ascending=False).head(10))

    if "capture_id" in hard_cases.columns:
        print("\nMisclassifications by Capture:")
        print(hard_cases["capture_id"].value_counts().head(5))


2026-03-27 13:05:06 | INFO | ai-vpn-firewall | Analyzing Hard Cases...

Number of Misclassified Test Samples: 830 out of 9384

Top Misclassified Samples (by error):
            capture_id dataset  label flow_id  p_xgb_raw  p_lgbm_raw  \
68605  ssh__chunk_0000  usbvpn      1   ssh_4   0.003845    0.004574   
68604  ssh__chunk_0000  usbvpn      1   ssh_3   0.003601    0.004457   
69509  ssh__chunk_0000  usbvpn      1   ssh_4   0.004095    0.002918   
69508  ssh__chunk_0000  usbvpn      1   ssh_3   0.001477    0.002907   
69154  ssh__chunk_0000  usbvpn      1   ssh_4   0.001118    0.003295   
69505  ssh__chunk_0000  usbvpn      1   ssh_0   0.001143    0.002911   
69507  ssh__chunk_0000  usbvpn      1   ssh_2   0.003392    0.003510   
68816  ssh__chunk_0000  usbvpn      1   ssh_2   0.012276    0.005550   
68603  ssh__chunk_0000  usbvpn      1   ssh_2   0.025272    0.006115   
69506  ssh__chunk_0000  usbvpn      1   ssh_1   0.017219    0.009911   

       p_cat_raw  prob_raw  prob_iso  prob

In [5]:
# 4. Leakage Check: Correlation with Label
# We check correlations on the FULL feature set (loaded earlier)
if has_features:
    logger.info("Checking for Linear Leakage (on full dataset)...")
    corrs = []
    # We need to ensure df_features has 'label'
    if "label" in df_features.columns:
        # Sample for speed if too large
        df_sample = df_features.sample(min(10000, len(df_features)), random_state=42)

        for col in feature_cols:
            if col in df_sample.columns:
                # Ensure numeric
                if pd.api.types.is_numeric_dtype(df_sample[col]):
                    c = df_sample[col].corr(df_sample["label"])
                    corrs.append((col, c))

        corr_df = pd.DataFrame(corrs, columns=["Feature", "Correlation"]).sort_values("Correlation", key=abs, ascending=False)
        print("\nTop Correlations with Label:")
        print(corr_df.head(10))

        if not corr_df.empty and abs(corr_df.iloc[0]["Correlation"]) > 0.95:
            print(f"\nWARNING: Feature '{corr_df.iloc[0]['Feature']}' has >0.95 correlation with label. Likely leakage.")
        else:
            print("\nNo single feature has >0.95 linear correlation with label.")
    else:
        print("Label column missing in features dataframe, skipping correlation check.")
else:
    print("Skipping Leakage Check (features not loaded).")


2026-03-27 13:05:06 | INFO | ai-vpn-firewall | Checking for Linear Leakage (on full dataset)...

Top Correlations with Label:
         Feature  Correlation
22    sz_all_std     0.367645
12  iat_mean_max    -0.286984
8    iat_all_std    -0.284039
7   iat_all_mean    -0.281007
11   iat_all_p75    -0.269977
13  iat_mean_min    -0.268847
14   iat_std_max    -0.256795
6     sz_std_min    -0.233403
5     sz_std_max    -0.232137
24    sz_all_p25    -0.224636

No single feature has >0.95 linear correlation with label.


C:\Users\scoti\PycharmProjects\ai-vpn-firewall\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\scoti\PycharmProjects\ai-vpn-firewall\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [6]:
# 5. Distribution of Top Feature
if has_features and 'imp_df' in locals() and imp_df is not None and not imp_df.empty:
    top_feat = imp_df.iloc[0]["Feature"]
    if top_feat in df_features.columns:
        plt.figure(figsize=(10, 6))
        # Use sample for plotting
        plot_sample = df_features.sample(min(5000, len(df_features)), random_state=42)
        sns.boxplot(x="label", y=top_feat, hue="dataset", data=plot_sample)
        plt.title(f"Distribution of '{top_feat}' by Label and Dataset")
        plt.show()
else:
    print("Skipping Feature Distribution Plot (features not loaded or importance not available).")

Skipping Feature Distribution Plot (features not loaded or importance not available).
